# Day 3 Lab: AI Governance in Practice

Evaluating AI architectures, diagnosing failure modes, and designing
graduated automation

> **Expected Time**
>
> -   Exercise 1 (Model comparison analysis): ≈ 30 minutes
> -   Exercise 2 (The confidence trap): ≈ 25 minutes
> -   Exercise 3 (Graduated automation design): ≈ 30 minutes
> -   Exercise 4 (Governance briefing note): ≈ 30 minutes
> -   Total: ≈ 115 minutes

<figure>
<a
href="https://colab.research.google.com/github/quinfer/minimba-notebooks/blob/main/lab03_ai_governance.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Setup (Colab-only installs)

In [ ]:
# Run this cell in Colab if a package is missing
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    print("Setup complete.")
except Exception:
    !pip -q install numpy pandas matplotlib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    print("Setup complete.")

## Before You Start

This lab uses real research findings from the UKFin+ project to practise
the governance frameworks from today’s lectures. Each exercise builds on
the previous one:

| Exercise | Day 3 concept | What you’ll do |
|-------------------|-------------------------|-----------------------------|
| 1 | Model comparison | Analyse the six-model results and recommend architectures |
| 2 | Confidence trap | Explore inverse calibration and its implications |
| 3 | Graduated automation | Design an automation routing framework |
| 4 | Governance note | Draft a professional governance recommendation |

All code is provided. Your job is to **interpret results, make judgement
calls, and write professional recommendations**.

# Exercise 1 — Model Comparison Analysis (30 min)

## Why This Matters

In practice, organisations must choose between competing AI
architectures. The UKFin+ study provides a rare controlled comparison —
six models evaluated on the same 36 rules under the same scoring regime.
Your task is to interpret these results as a decision-maker, not just a
technician.

## Task 1a: Examine the Results

In [ ]:
# UKFin+ canonical 36-rule evaluation results
results = pd.DataFrame({
    'Model': ['GraphDB (symbolic)', 'Few-Shot (M1)', 'GraphRAG k=10 (M2a)',
              'GraphRAG k=50 (M2b)', 'Logic-LM (M3)', 'Schema-Enforced (M4)',
              'Chain-of-Thought (M5)', 'Ambiguity-Aware (M6)'],
    'Architecture': ['Symbolic', 'Neural', 'Neural+RAG', 'Neural+RAG',
                     'Hybrid', 'Neural+Schema', 'Neural+CoT', 'Hybrid+Routing'],
    'GED_Similarity': [88.0, 73.9, 72.9, 71.9, 72.7, 73.9, 54.0, 73.9],
    'Lexical_Consistency': [94.0, 2.8, 1.9, 2.8, 2.8, 1.9, 3.7, 1.9],
    'Confidence': [np.nan, 89.9, 89.9, 90.3, 89.8, 89.9, 88.8, 89.9],
    'Calibration_r': [np.nan, -0.545, -0.469, -0.500, 0.135, -0.352, 0.190, -0.352],
    'Cost_per_rule': [np.nan, 0.0024, 0.0030, 0.0035, 0.0015, 0.0028, 0.0065, 0.0030],
})

# Display the full comparison table
print("=" * 90)
print("UKFin+ Canonical 36-Rule Evaluation: Full Results")
print("=" * 90)
print(results.to_string(index=False))
print("\n")

# Highlight key contrasts
print("KEY CONTRASTS:")
print(f"  Structural gap (symbolic vs best LLM): {88.0 - 73.9:.1f} percentage points")
print(f"  Lexical gap (symbolic vs best LLM): {94.0 - 3.7:.1f} percentage points")
print(f"  Cost range (LLMs only): ${results['Cost_per_rule'].min():.4f} - ${results['Cost_per_rule'].max():.4f} per rule")
print(f"  Models with inverse calibration: {(results['Calibration_r'].dropna() < 0).sum()} of {results['Calibration_r'].dropna().shape[0]}")

## Task 1b: Visualise the Trade-offs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Structure vs Lexical
llm_mask = results['Architecture'] != 'Symbolic'
ax = axes[0]
ax.scatter(results.loc[~llm_mask, 'GED_Similarity'],
           results.loc[~llm_mask, 'Lexical_Consistency'],
           s=200, color='#2ca02c', zorder=5, label='Symbolic')
ax.scatter(results.loc[llm_mask, 'GED_Similarity'],
           results.loc[llm_mask, 'Lexical_Consistency'],
           s=120, color='#4C72B0', zorder=5, label='LLM-based')
for _, row in results.iterrows():
    ax.annotate(row['Model'].split(' (')[0], (row['GED_Similarity'], row['Lexical_Consistency']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Structural Accuracy (GED %)')
ax.set_ylabel('Lexical Consistency (%)')
ax.set_title('Structure vs Terminology')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Accuracy vs Cost (LLMs only)
llm_data = results[llm_mask].dropna(subset=['Cost_per_rule'])
colours_cal = ['#d62728' if r < 0 else '#2ca02c' for r in llm_data['Calibration_r']]
ax = axes[1]
ax.scatter(llm_data['Cost_per_rule'] * 1000, llm_data['GED_Similarity'],
           s=120, c=colours_cal, edgecolors='white', linewidths=1.5, zorder=5)
for _, row in llm_data.iterrows():
    ax.annotate(row['Model'].split(' (')[0], (row['Cost_per_rule'] * 1000, row['GED_Similarity']),
                textcoords='offset points', xytext=(5, 5), fontsize=7)
ax.set_xlabel('Cost per Rule ($ × 1000)')
ax.set_ylabel('Structural Accuracy (GED %)')
ax.set_title('Accuracy vs Cost\n(Red = inverse calibration)')
ax.grid(alpha=0.3)

# Panel 3: Model selection matrix
use_cases = ['Research\nBaseline', 'Cost-\nSensitive', 'Schema\nEnforcement',
             'Regulatory\nAudit', 'Production\nDeployment']
recommended = ['M1\nFew-Shot', 'M2\nGraphRAG', 'M4\nSchema', 'M5\nCoT', 'M6\nAmbiguity']
priorities = ['Simplicity', 'Cost', 'Lexical\nAccuracy', 'Explain-\nability', 'Balance']

ax = axes[2]
ax.set_xlim(0, 6)
ax.set_ylim(0, 6)
for i, (uc, rec, pri) in enumerate(zip(use_cases, recommended, priorities)):
    y = 5 - i
    ax.text(0.5, y, uc, fontsize=8, va='center', fontweight='bold')
    ax.text(2.8, y, rec, fontsize=8, va='center', color='#4C72B0', fontweight='bold')
    ax.text(4.8, y, pri, fontsize=8, va='center', color='#666')
    ax.axhline(y - 0.4, color='#eee', linewidth=0.5)

ax.text(0.5, 5.6, 'Use Case', fontsize=9, fontweight='bold')
ax.text(2.8, 5.6, 'Model', fontsize=9, fontweight='bold')
ax.text(4.8, 5.6, 'Priority', fontsize=9, fontweight='bold')
ax.axis('off')
ax.set_title('Model Selection Matrix')

plt.tight_layout()
plt.show()

### Questions (Write 150–200 words)

1.  You are advising a mid-sized asset management firm (500 fund
    classifications to process annually). Which model(s) would you
    recommend and why?
2.  The structural gap between symbolic and LLM approaches is ~15
    percentage points, but the lexical gap is ~91 percentage points.
    Which gap matters more for production deployment? Why?
3.  Chain-of-Thought (M5) has the best calibration but worst structural
    accuracy. Under what specific circumstances would you choose it
    despite the accuracy penalty?

# Exercise 2 — The Confidence Trap (25 min)

## Why This Matters

Many AI deployment proposals rely on confidence-based routing: “if the
model is confident, automate; if uncertain, escalate.” The UKFin+
evidence shows this can be systematically unsafe. This exercise makes
the danger concrete.

## Task 2a: Simulate Confidence-Based vs Ambiguity-Based Routing

In [ ]:
np.random.seed(42)
n_rules = 100

# Simulate rules with varying accuracy and confidence
# For inversely calibrated model: high confidence → LOW accuracy
accuracy = np.random.beta(3, 2, size=n_rules)  # Skewed towards higher accuracy
confidence_inverse = 1 - accuracy + np.random.normal(0, 0.1, size=n_rules)  # Inverse relationship
confidence_inverse = np.clip(confidence_inverse, 0.5, 1.0)

# For well-calibrated model: high confidence → HIGH accuracy
confidence_calibrated = accuracy + np.random.normal(0, 0.1, size=n_rules)
confidence_calibrated = np.clip(confidence_calibrated, 0.5, 1.0)

# Ambiguity score (independent of model performance)
ambiguity = np.random.beta(2, 3, size=n_rules)  # Most rules are low-medium ambiguity

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Inversely calibrated (dangerous)
ax = axes[0]
sc = ax.scatter(confidence_inverse, accuracy, c=accuracy, cmap='RdYlGn',
                s=40, alpha=0.7, edgecolors='white', linewidths=0.5)
ax.axvline(0.85, color='red', linestyle='--', label='Auto-approve threshold')
ax.set_xlabel('Model Confidence')
ax.set_ylabel('Actual Accuracy')
ax.set_title('Inverse Calibration (4 of 6 models)\nHigh confidence → LOW accuracy')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Highlight the danger zone
danger = (confidence_inverse > 0.85) & (accuracy < 0.5)
ax.scatter(confidence_inverse[danger], accuracy[danger], s=100,
           facecolors='none', edgecolors='red', linewidths=2, zorder=10)
n_danger = danger.sum()
ax.text(0.87, 0.2, f'{n_danger} rules\nauto-approved\nbut WRONG',
        fontsize=8, color='red', fontweight='bold')

# Panel 2: Well-calibrated (safe)
ax = axes[1]
ax.scatter(confidence_calibrated, accuracy, c=accuracy, cmap='RdYlGn',
           s=40, alpha=0.7, edgecolors='white', linewidths=0.5)
ax.axvline(0.85, color='green', linestyle='--', label='Auto-approve threshold')
ax.set_xlabel('Model Confidence')
ax.set_ylabel('Actual Accuracy')
ax.set_title('Well-Calibrated (ideal)\nHigh confidence → HIGH accuracy')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: Ambiguity-based routing (safe regardless of calibration)
ax = axes[2]
ax.scatter(ambiguity, accuracy, c=accuracy, cmap='RdYlGn',
           s=40, alpha=0.7, edgecolors='white', linewidths=0.5)
ax.axvline(0.3, color='green', linestyle='--', label='Auto-approve threshold')
ax.axvline(0.6, color='orange', linestyle='--', label='Hybrid threshold')
ax.set_xlabel('Text Ambiguity Score')
ax.set_ylabel('Actual Accuracy')
ax.set_title('Ambiguity-Based Routing\nIndependent of model calibration')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify the routing outcomes
print("ROUTING OUTCOMES:")
print(f"\nConfidence-based routing (threshold > 0.85):")
auto_conf = confidence_inverse > 0.85
print(f"  Auto-approved: {auto_conf.sum()} rules")
print(f"  Average accuracy of auto-approved: {accuracy[auto_conf].mean():.1%}")
print(f"  Rules with accuracy < 50% that were auto-approved: {((auto_conf) & (accuracy < 0.5)).sum()}")

print(f"\nAmbiguity-based routing (threshold < 0.3):")
auto_amb = ambiguity < 0.3
print(f"  Auto-approved: {auto_amb.sum()} rules")
print(f"  Average accuracy of auto-approved: {accuracy[auto_amb].mean():.1%}")
print(f"  Rules with accuracy < 50% that were auto-approved: {((auto_amb) & (accuracy < 0.5)).sum()}")

## Task 2b: Quantify the Cost of Getting Routing Wrong

In [ ]:
# Costs
cost_auto = 0.003      # $ per rule (AI only)
cost_hybrid = 40       # $ per rule (AI + 15 min human review at $150/hr)
cost_human = 150       # $ per rule (1 hour human)
cost_error = 5000      # $ per undetected error (regulatory risk)

# Confidence-based routing
n_auto_conf = auto_conf.sum()
n_review_conf = (~auto_conf).sum()
errors_conf = ((auto_conf) & (accuracy < 0.5)).sum()
total_conf = (n_auto_conf * cost_auto + n_review_conf * cost_hybrid + errors_conf * cost_error)

# Ambiguity-based routing
n_auto_amb = auto_amb.sum()
n_hybrid_amb = ((ambiguity >= 0.3) & (ambiguity < 0.6)).sum()
n_human_amb = (ambiguity >= 0.6).sum()
errors_amb = ((auto_amb) & (accuracy < 0.5)).sum()
total_amb = (n_auto_amb * cost_auto + n_hybrid_amb * cost_hybrid +
             n_human_amb * cost_human + errors_amb * cost_error)

# Full manual baseline
total_manual = n_rules * cost_human

print("COST COMPARISON FOR 100 RULES:")
print(f"\n  Full manual:           ${total_manual:,.0f}")
print(f"  Confidence routing:    ${total_conf:,.0f}  (errors: {errors_conf}, error cost: ${errors_conf * cost_error:,.0f})")
print(f"  Ambiguity routing:     ${total_amb:,.0f}  (errors: {errors_amb}, error cost: ${errors_amb * cost_error:,.0f})")
print(f"\n  Confidence saving vs manual: {(1 - total_conf/total_manual):.0%}")
print(f"  Ambiguity saving vs manual:  {(1 - total_amb/total_manual):.0%}")

### Questions (Write 150–200 words)

1.  Which routing strategy produces lower total cost (including error
    costs)? Why?
2.  If the cost of an undetected error were £50,000 instead of £5,000
    (e.g., regulatory fine), how would that change the comparison?
3.  A vendor tells you: “Our model is 90% confident on 80% of rules.”
    Based on the UKFin+ evidence, is this reassuring or alarming? Why?

# Exercise 3 — Graduated Automation Design (30 min)

## Why This Matters

You are now designing a graduated automation framework for your
organisation. This is the practical output that matters most —
translating research evidence into deployment decisions.

## Task 3a: Classify Rules by Automation Level

In [ ]:
# Sample regulatory rules with varying characteristics
rules = pd.DataFrame({
    'Rule': [
        'Fund must allocate at least 80% to government bonds',
        'Fund must be predominantly invested in equities',
        'Investment must be appropriate for fund objectives',
        'Maximum 10% in any single issuer',
        'Reasonable diversification across sectors',
        'Fund must not hold more than 5% in unlisted securities',
        'Investments must be consistent with published policy',
        'Minimum 70% in investment-grade fixed income',
        'Fund may hold ancillary liquid assets',
        'Material changes require investor notification',
    ],
    'Ambiguity_Keywords': [0, 1, 2, 0, 1, 0, 2, 0, 1, 1],
    'Quantitative': [True, False, False, True, False, True, False, True, False, False],
    'Constraint_Count': [1, 1, 0, 1, 0, 1, 0, 1, 0, 0],
})

# Classify
def classify_rule(row):
    if row['Ambiguity_Keywords'] == 0 and row['Quantitative']:
        return 'Automatic'
    elif row['Ambiguity_Keywords'] <= 1:
        return 'Hybrid'
    else:
        return 'Human-led'

rules['Recommended_Level'] = rules.apply(classify_rule, axis=1)

# Display
colours_map = {'Automatic': '#27ae60', 'Hybrid': '#f39c12', 'Human-led': '#e74c3c'}

print("RULE CLASSIFICATION:")
print("=" * 100)
for _, row in rules.iterrows():
    level = row['Recommended_Level']
    quant = 'Quantitative' if row['Quantitative'] else 'Qualitative'
    amb = ['None', 'Some', 'High'][row['Ambiguity_Keywords']]
    print(f"  [{level:10s}] {row['Rule']}")
    print(f"              → {quant}, Ambiguity: {amb}, Constraints: {row['Constraint_Count']}")
    print()

# Summary
counts = rules['Recommended_Level'].value_counts()
print("ROUTING SUMMARY:")
for level in ['Automatic', 'Hybrid', 'Human-led']:
    n = counts.get(level, 0)
    print(f"  {level}: {n} rules ({n/len(rules):.0%})")

### Task 3b: Your Classification

Review the 10 rules above. For each:

1.  Do you agree with the automated classification? If not, why would
    you change it?
2.  Identify one rule where the classification is most uncertain. What
    additional information would help you decide?
3.  If you had to err on one side (over-automate or under-automate),
    which would you choose for regulatory compliance? Why?

### Questions (Write 200–250 words)

1.  The UKFin+ research found that 60% of FCA rules contain ambiguous
    language. Does the sample above reflect this? If it were a real
    production system, how would you calibrate the ambiguity thresholds?
2.  Rule 2 (“predominantly invested in equities”) is classified as
    Hybrid. But “predominantly” means different things to different
    people (60%? 70%? 80%?). How should the AI system handle this?
3.  If the FCA updated a rule from “reasonable diversification” to “at
    least four distinct sectors,” how would this change the automation
    level? What does this tell you about the relationship between
    regulatory drafting style and automation potential?

# Exercise 4 — Governance Briefing Note (30 min)

## No Code Required

You are writing a **one-page governance note** (400–600 words) for your
firm’s board, recommending whether and how to deploy AI for regulatory
rule extraction. Use the evidence from today’s exercises.

**Your note should address:**

**1. Recommendation** (one paragraph) State clearly whether you
recommend deployment, and at what level. Use the graduated automation
framework.

**2. Evidence Base** (one paragraph) Summarise the UKFin+ findings:
structural landscape (88% vs 74%), lexical drift (94% vs 2–4%), inverse
calibration (4/6 models), and the 40/20/40 routing split. Acknowledge
the sample size limitation (36 rules) and the need for production
validation.

**3. Risk Assessment** (one paragraph) Identify the top three risks,
drawing on today’s evidence:

-   Lexical drift → outputs don’t integrate with existing systems
-   Confidence trap → automated routing is systematically unsafe
-   Multi-constraint extraction → compound rules are unreliable

For each risk, state the mitigation (schema enforcement, ambiguity-based
routing, human review).

**4. Governance Controls** (bullet points) Specify the non-negotiables:

-   Vocabulary governance (curated knowledge base, schema enforcement)
-   Independent routing (ambiguity-based, not confidence-based)
-   Selective explainability (where legally required)
-   Continuous validation (periodic revalidation on new regulatory
    texts)

**5. Cost-Benefit** (one paragraph) Provide a conditional projection: if
the 40/20/40 routing holds, projected savings of 40–50%. State the
conditions required for this projection to hold. Apply Day 2’s evidence
discipline to your own cost claims.

> **Connection to Assessment**
>
> This exercise is direct preparation for the final briefing note. The
> assessment asks you to evaluate a real FinTech or AI claim — the
> governance note you write here demonstrates exactly the judgement we
> are looking for.

# Synthesis

## Connecting the Exercises

| Exercise | Key lesson | Day 3 framework |
|--------------------|----------------------|-------------------------------|
| 1 (Model comparison) | No single “best” model; choice depends on use case and priorities | Architecture trade-offs |
| 2 (Confidence trap) | Model self-assessment can be systematically misleading | Route by observable properties, not confidence |
| 3 (Graduated automation) | Ambiguity is measurable and can drive routing decisions | Match automation to specification level |
| 4 (Governance note) | Professional recommendations require evidence, risk assessment, and controls | Evidence-based governance |

## Final Reflection (150–200 words)

Considering all three days of the programme, write a short reflection:

1.  Before this programme, how would you have evaluated an AI vendor’s
    compliance product? What would you do differently now?
2.  Which framework from the three days do you find most useful for your
    professional context? (Economics of prediction, evidence discipline,
    graduated automation, or the evaluation checklist?)
3.  The UKFin+ research found that AI is **closer to human performance
    on structure** (~15% gap) than people expect, but **much worse on
    consistency** (~91% gap). What does this tell us about where AI
    creates value and where it creates risk?

## References